In [2]:
!pip install kafka-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 603.6/603.6 kB 6.8 MB/s eta 0:00:00a 0:00:01


In [15]:
from kafka import KafkaConsumer
import json
from datetime import datetime

consumer = KafkaConsumer(
    'crypto-stream',
    bootstrap_servers=['kafka:9092'],
    auto_offset_reset='latest',
    enable_auto_commit=True,
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

total_records = 0
crypto_stats = {}  

print("Konsument Kafki uruchomiony...", flush=True)
print("Trwa analiza strumienia z Binance w czasie rzeczywistym...\n", flush=True)
print(f"{'TIMESTAMP':<20} | {'SYMBOL':<8} | {'PRICE':<10} | {'AVG PRICE':<10} | {'MAX PRICE':<10} | {'COUNT':<5}", flush=True)
print("-" * 75, flush=True)

try:
    for message in consumer:
        data = message.value
        
        raw_timestamp = data.get('timestamp')
        symbol = data.get('symbol')
        price = data.get('price')
        
        if not raw_timestamp or not symbol or price is None or price <= 0:
            continue
            
        total_records += 1
        readable_timestamp = datetime.fromtimestamp(raw_timestamp).strftime('%H:%M:%S')
        
        if symbol not in crypto_stats:
            crypto_stats[symbol] = {
                'total_price': 0.0,
                'count': 0,
                'max_price': 0.0
            }
            
        crypto_stats[symbol]['total_price'] += price
        crypto_stats[symbol]['count'] += 1
        if price > crypto_stats[symbol]['max_price']:
            crypto_stats[symbol]['max_price'] = price
            
        avg_price = crypto_stats[symbol]['total_price'] / crypto_stats[symbol]['count']
        max_price = crypto_stats[symbol]['max_price']
        current_count = crypto_stats[symbol]['count']
        
        print(f"{readable_timestamp:<20} | {symbol:<8} | {price:<10.2f} | {avg_price:<10.2f} | {max_price:<10.2f} | {current_count:<5}", flush=True)

except KeyboardInterrupt:
    print("\n Zatrzymano zaawansowaną analizę strumieniową.", flush=True)
    print(f" Łącznie podczas sesji przetworzono poprawnie rekordów: {total_records}", flush=True)

Konsument Kafki uruchomiony...
Trwa analiza strumienia z Binance w czasie rzeczywistym...

TIMESTAMP            | SYMBOL   | PRICE      | AVG PRICE  | MAX PRICE  | COUNT
---------------------------------------------------------------------------


/tmp/ipykernel_636/3249591088.py:5: DeprecationWarning: value_deserializer does not implement kafka.serializer.Deserializer
  consumer = KafkaConsumer(



 Zatrzymano zaawansowaną analizę strumieniową.
 Łącznie podczas sesji przetworzono poprawnie rekordów: 0
